<a href="https://colab.research.google.com/github/suryavedula/AgentCon-Cloud-Repo/blob/main/IMDB_Sentiment_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis: From Bag-of-Words to Neural Classifiers

---

> ### 📋 Assignment
> You will build a complete sentiment classification pipeline on the **IMDB Movie Review** dataset — 50,000 reviews labelled positive or negative.
>
> **Part 1** uses classical machine learning on **binary one-hot vectors** — the simplest possible text representation.
> **Part 2** introduces neural sequence models for NLP. The boilerplate (data loading, vocabulary, training loop, evaluation) is **fully provided and pre-written**. Your job is to:
> 1. Build the **model architectures** (`### CODE HERE ###` inside `nn.Module` classes)
> 2. Write **interpretation and reflection** cells
>
> **Cells marked `# ── PROVIDED ──` carry no marks** — just run them.
> **Cells marked `### CODE HERE ###` carry marks** — these are yours to complete.
>
> Use `random_state=42` wherever accepted.

---

## What You Will Build

| Part | Approach | Key Idea |
|------|----------|----------|
| **Part 1** | Classical ML — LR, Naive Bayes, SVM, Random Forest | Binary one-hot vectors (word present/absent) |
| **Part 2A** | Vanilla RNN | Reads tokens sequentially; hidden state carries context |
| **Part 2B** | LSTM | Gated memory solves long-range forgetting |
| **Part 2C** | Bidirectional LSTM + FastText embeddings | Both-direction context + pre-trained word meaning |
| **Part 2D** | Cross-Attention Classifier | Every token attends to every other — no sequential bottleneck |

**Total Marks: 100** (Part 1: 45 | Part 2: 55)

### The Central Question
> *Can a model reading `"the film is not as bad as critics claimed"` correctly identify it as **positive**?*
>
> A one-hot vector sees `bad` and `critics` as present — and guesses *negative*. A neural model reading the full sentence in order has a fighting chance. An attention model can show you exactly which words it focused on.

---

# PART 1: Classical Sentiment Classification

## Context

The IMDB dataset contains 50,000 movie reviews — 25,000 for training, 25,000 for testing, perfectly balanced between positive and negative.

Reviews average **~230 words** — long enough that word order and negation genuinely matter. In this part, you will represent each review as a **binary one-hot vector**: a vector of 0s and 1s where a 1 simply means a word appeared in the review, and 0 means it did not. Frequency is ignored entirely — `"masterpiece"` appearing once and `"bad"` appearing ten times both become 1.

This is intentionally the simplest possible text representation. It sets a realistic ceiling that the neural models in Part 2 need to meaningfully exceed. You will see exactly where this representation breaks down.

## Stage 1: Data Loading  <font color="red">**[4 marks]**</font>

**What to do:**
- The libraries below are pre-imported. Add anything else you need.
- Load the IMDB dataset using `load_dataset('imdb')` from HuggingFace.
- Extract train texts, test texts, and their labels into plain Python lists.
- Print the number of examples and class distribution.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re, warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, confusion_matrix, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

In [ ]:
# Load the IMDB dataset

dataset     = ### CODE HERE ###
train_texts  = ### CODE HERE ###
train_labels = ### CODE HERE ###
test_texts   = ### CODE HERE ###
test_labels  = ### CODE HERE ###
y_train = np.array(train_labels)
y_test  = np.array(test_labels)

In [ ]:
# Print number of examples and label distribution
### CODE HERE ###

## Stage 2: Data Understanding  <font color="red">**[8 marks]**</font>

### 2.1 Class Distribution  <font color="red">[1 mark]</font>

Plot a bar chart of positive vs negative counts in the training set.

In [ ]:
### CODE HERE ###

### 2.2 Review Length Distribution  <font color="red">[2 marks]</font>

Compute word count per training review. Plot a histogram and print the mean, median, and 95th percentile length.
Also plot lengths **separately for positive and negative reviews** using overlapping histograms.

In [ ]:
### CODE HERE ###

### 2.3 Sample Reviews  <font color="red">[1 mark]</font>

Print the first 200 characters of one positive and one negative review. Note any HTML artefacts you see.

In [ ]:
### CODE HERE ###

### 2.4 Top Words by Class  <font color="red">[2 marks]</font>

Fit a `CountVectorizer` (with `stop_words='english'`) on training data only.
Plot the top 20 most frequent words for positive reviews and negative reviews side by side.

In [ ]:
### CODE HERE ###

### 2.5 Negation Examples  <font color="red">[2 marks]</font>

Find 5 training reviews where `"not"` appears at least 3 times. Print the first 50 words of each and their true label.

> These are the examples classical models will get wrong most often. Keep them in mind for Stage 6.

In [ ]:
### CODE HERE ###

### ✍️ Interpretation Checkpoint

**[Your Answer]:** Answer each question in 2–3 sentences.

1. Is the dataset balanced? Does this affect our choice of F1 vs accuracy?
2. The top words for positive and negative look nearly identical. Given that one-hot vectors give all words equal weight, why does this make sentiment classification especially hard?
3. Why is review length a specific problem for vanilla RNNs? (Think about what happens to gradients after 200+ backpropagation steps.)
4. Why should you remove `<br />` HTML tags before both one-hot vectorisation and neural models?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

3. ***your answer here***

4. ***your answer here***

## Stage 3: Preprocessing and Feature Engineering  <font color="red">**[6 marks]**</font>

## Binary One-Hot Vectors: The Simplest Text Representation

A **binary one-hot vector** represents a document as a fixed-length vector of 0s and 1s:

- **1** = this word appeared at least once in the review
- **0** = this word did not appear

For a vocabulary of 50,000 words, each review becomes a vector of 50,000 numbers, almost all of which are 0.

```python
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(binary=True, max_features=50000)
# binary=True → all non-zero counts become 1
# Each row: 1 if word present, 0 if absent — frequency ignored
```

**What one-hot cannot represent:**
- **Frequency** — `"bad"` appearing 10 times looks identical to `"bad"` appearing once
- **Rarity** — `"masterpiece"` (rare, diagnostic) has the same weight as `"film"` (common, uninformative)
- **Order** — `"not bad"` and `"bad not"` produce identical vectors

This is a deliberately weak representation. The classical models in Stage 4 will plateau around **80–84% F1** — well below what TF-IDF would give — leaving clear room for neural models to improve.

**Critical rule:** Fit the vectoriser on **training data only**. Fitting on test data leaks vocabulary statistics into training.

### 3.1 Text Cleaning  <font color="red">[2 marks]</font>

Write a `clean_text(text)` function that:
- Removes HTML tags (`<br />`, `<b>`, etc.) using a regex
- Lowercases the text
- Strips leading/trailing whitespace

Apply it to all train and test texts. Print a before/after example to confirm it works.

In [ ]:
def clean_text(text):
    ### CODE HERE ###
    pass

# Apply to train and test
### CODE HERE ###

# Print one before/after example
### CODE HERE ###

### 3.2 Build One-Hot Features  <font color="red">[4 marks]</font>

Create a `CountVectorizer` with:
- `binary=True` — converts all counts to 0/1
- `max_features=50000` — keep the 50,000 most frequent words
- `min_df=5` — ignore words appearing in fewer than 5 reviews
- `stop_words='english'`

**Fit on training data only**, then transform both splits. Print the shape and sparsity of the resulting matrices.

> **Note:** No `ngram_range=(1,2)` this time — with one-hot, bigrams would double the already-large feature space with minimal benefit. Unigrams only.

In [ ]:
### CODE HERE ###

## Helper: Evaluation Function

Define a reusable `evaluate_model(name, y_true, y_pred)` function. It should:
- Print accuracy, F1, precision, recall
- Print a `classification_report`
- Plot a confusion matrix heatmap
- Return the metrics as a dict (for building a comparison table later)

In [ ]:
def evaluate_model(name, y_true, y_pred):
    ### CODE HERE ###
    pass

## Using Pipeline + GridSearchCV for Text

Chain the vectoriser and classifier in a `Pipeline`, then wrap in `GridSearchCV`. This ensures the vectoriser is re-fit on each cross-validation fold — preventing data leakage.

```python
pipe = Pipeline([
    ('vec', CountVectorizer(binary=True, min_df=5, stop_words='english')),
    ('clf', LogisticRegression(random_state=42))
])
param_grid = {
    'vec__max_features': [20000, 50000],   # prefix is the pipeline step name
    'clf__C':            [0.1, 1, 10]
}
grid = GridSearchCV(pipe, param_grid, scoring='f1', cv=5, n_jobs=-1)
grid.fit(train_texts_clean, y_train)   # pass CLEANED TEXT — not the matrix
```

> **Important:** Pass cleaned raw text to `grid.fit()`. The pipeline applies the vectoriser internally at each fold, which is what prevents leakage.

## Stage 4: Model Training, Tuning, and Evaluation  <font color="red">**[20 marks]**</font>

Train **four classifiers**. For each: build a Pipeline with a `CountVectorizer(binary=True, ...)`, run GridSearchCV with `scoring='f1'` and `cv=5`, print best params and CV score, then evaluate on the test set using `evaluate_model()`.

> **Expected F1 range:** ~78–84%. One-hot vectors give no frequency or rarity information, so classical models cannot achieve the same ceiling as TF-IDF. This is deliberate — it leaves meaningful room for neural models to improve.

### 4.1 Logistic Regression  <font color="red">[5 marks]</font>

Hyperparameters to tune:
- `vec__max_features`: `[20000, 50000]`
- `clf__C`: `[0.01, 0.1, 1, 10]`

Set `max_iter=1000`, `random_state=42`.

In [ ]:
### CODE HERE ###

In [ ]:
# Predict on test set and evaluate
### CODE HERE ###

### 4.2 Bernoulli Naive Bayes  <font color="red">[5 marks]</font>

Use `BernoulliNB` — the variant of Naive Bayes designed specifically for **binary features**. It models each word as a Bernoulli (present/absent) trial, which matches our one-hot representation exactly.

Hyperparameters to tune:
- `vec__max_features`: `[20000, 50000]`
- `clf__alpha`: `[0.01, 0.1, 0.5, 1.0]`

> **Why BernoulliNB and not ComplementNB?** `ComplementNB` is designed for count features. `BernoulliNB` is the correct choice for binary presence/absence data — it penalises words that are *absent* from a class, which is meaningful when features are truly binary.

In [ ]:
### CODE HERE ###

In [ ]:
### CODE HERE ###

### 4.3 LinearSVC  <font color="red">[5 marks]</font>

Hyperparameters to tune:
- `vec__max_features`: `[20000, 50000]`
- `clf__C`: `[0.01, 0.1, 1, 10]`

Set `random_state=42`, `max_iter=2000`.

In [ ]:
### CODE HERE ###

In [ ]:
### CODE HERE ###

### 4.4 Random Forest  <font color="red">[5 marks]</font>

Random Forest builds many decision trees on random subsets of features and takes a majority vote. It handles high-dimensional sparse binary vectors well and naturally produces **feature importances** — making it ideal for Stage 5.3.

Hyperparameters to tune:
- `vec__max_features`: `[20000, 50000]`
- `clf__n_estimators`: `[100, 200]`
- `clf__max_depth`: `[None, 20, 50]`
- `clf__min_samples_leaf`: `[1, 2]`

Set `random_state=42`, `n_jobs=-1`.

> **Note:** Random Forest is slower than LR or SVM on this dataset — expect ~2–3 minutes. This is normal.

In [ ]:
### CODE HERE ###

In [ ]:
### CODE HERE ###

## Stage 5: Classical Model Comparison  <font color="red">**[7 marks]**</font>

### 5.1 Comparison Table  <font color="red">[2 marks]</font>

Build a DataFrame comparing all four models on Accuracy, Precision, Recall, and F1. Sort by F1 descending.

In [ ]:
### CODE HERE ###

### 5.2 Visualise  <font color="red">[2 marks]</font>

Grouped bar chart comparing Precision, Recall, and F1 across all four models.

In [ ]:
### CODE HERE ###

### 5.3 Most Important Features  <font color="red">[3 marks]</font>

From the **Random Forest** pipeline, extract the top 20 most important words using `.feature_importances_`.

Plot them as a horizontal bar chart sorted by importance.

> **Note:** Random Forest importance is non-negative (unlike LR/SVM coefficients) — you cannot separate positive/negative direction. Instead, these are the words that most reduced impurity across all trees, regardless of direction. The most important words should be strong sentiment markers.

In [ ]:
### CODE HERE ###

## Stage 6: Part 1 Reflection  <font color="red">**[8 marks]**</font>

**[Your Answer]:** Answer each question. Cite specific numbers from your results.

1. What is the F1 ceiling for classical models using one-hot vectors? How does this compare to what TF-IDF typically achieves (~87–89%)? What specific information does TF-IDF have that one-hot vectors lack?
2. Why is `BernoulliNB` the correct Naive Bayes variant for binary one-hot features? What would happen if you used `MultinomialNB` instead?
3. Find **3 reviews your best model got wrong**. Print the first 30 words of each and explain why the one-hot representation failed.
4. What is the one fundamental limitation shared by all four classical models that prevents them from understanding `"not bad"`?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

3. ***your answer here***

4. ***your answer here***

In [ ]:
# Find and print 3 misclassified reviews from your best model
### CODE HERE ###

---
# PART 2: Neural Sentiment Classifiers

## Context

Classical models with one-hot vectors have three fundamental weaknesses:
1. **No frequency** — `"bad"` appearing once = `"bad"` appearing ten times
2. **No rarity signal** — `"film"` (meaningless) has the same weight as `"masterpiece"` (highly diagnostic)
3. **No word order** — `"not bad"` and `"bad not"` are identical vectors

Neural sequence models address all three. They learn dense word representations (embeddings) that encode meaning, and process tokens sequentially or with attention to capture context.

## Your role in Part 2

All the boilerplate is written for you:
- Data pipeline (tokeniser, vocabulary, dataset class, dataloader)
- Training loop
- Evaluation function
- Positional encoding and attention mask utilities
- FastText embedding loader
- Attention visualisation

**You only write the model architectures and the interpretation cells.**

Read each concept box before building each model — they explain the new idea the architecture introduces.

## New NLP Concepts for Part 2

You know neural networks. Here is what is new when applying them to text:

### 1. Tokenisation and vocabulary
Instead of TF-IDF vectors, we convert each review to a **sequence of integer IDs**:
```
"not bad"  →  tokenise  →  ['not', 'bad']  →  map to IDs  →  [45, 312]
```
A **vocabulary** maps every known training word to a unique integer. Unknown words → `<UNK>` (ID 1). The padding token `<PAD>` (ID 0) is used to fill short sequences.

### 2. Padding and truncation
All sequences in a batch must be the same length. We fix every review to `MAX_LEN = 400`:
- Reviews longer than 400 tokens → keep the first 400 (truncate)
- Reviews shorter than 400 tokens → append zeros on the right (pad)

### 3. Embedding layer
`nn.Embedding(vocab_size, dim)` is a learnable lookup table. Row $i$ is the vector for word ID $i$. Unlike TF-IDF (sparse, one feature per word), embeddings are **dense** (e.g. 100 floats per word) and **learned from data**.
```python
self.embedding = nn.Embedding(vocab_size, 128, padding_idx=0)
# padding_idx=0 → PAD tokens get a zero vector and don't affect gradients
```

### 4. Binary classification output
The model outputs **one logit per review** (not two — this is binary). Loss is `BCEWithLogitsLoss`.
Prediction: `sigmoid(logit) > 0.5` → class 1 (positive), else class 0 (negative).

## Stage 1: Setup  *(provided — run all cells, no marks)*

All cells in this stage are fully written. Run them in order, read the comments, and make sure you understand what each part does before moving to Stage 2A.

In [ ]:
# ── PROVIDED ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  device: {device}')
if device.type == 'cpu':
    print('Tip: enable a GPU runtime in Google Colab (Runtime → Change runtime type → T4 GPU)')

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# ── PROVIDED ──────────────────────────────────────────────────────────────
# TOKENISER: strips HTML, lowercases, removes punctuation, splits on spaces.
def tokenize(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

# VOCABULARY: built from training data only.
# Words appearing fewer than MIN_FREQ times are treated as unknown (<UNK>).
MIN_FREQ = 5
counter  = Counter()
for doc in train_texts:
    counter.update(tokenize(doc))

word2idx = {'<PAD>': 0, '<UNK>': 1}
for word, freq in counter.items():
    if freq >= MIN_FREQ:
        word2idx[word] = len(word2idx)

idx2word   = {v: k for k, v in word2idx.items()}
VOCAB_SIZE = len(word2idx)
print(f'Vocabulary: {VOCAB_SIZE:,} words  (words seen < {MIN_FREQ}× → <UNK>)')

In [ ]:
# ── PROVIDED ──────────────────────────────────────────────────────────────
# DATASET: converts a raw review + label into a fixed-length integer tensor.
MAX_LEN    = 400   # reviews longer than this are truncated; shorter ones are padded
BATCH_SIZE = 64

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len):
        self.labels = labels
        self.seqs   = []
        for doc in texts:
            ids  = [word2idx.get(t, 1) for t in tokenize(doc)]  # unknown → 1
            ids  = ids[:max_len]                                  # truncate
            ids += [0] * (max_len - len(ids))                    # pad with 0
            self.seqs.append(ids)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (torch.tensor(self.seqs[idx],   dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.float))

train_ds = IMDBDataset(train_texts, y_train, word2idx, MAX_LEN)
test_ds  = IMDBDataset(test_texts,  y_test,  word2idx, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)
print(f'Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}')

In [ ]:
# ── PROVIDED ──────────────────────────────────────────────────────────────
# TRAINING LOOP — do not modify.
def train_model(model, dataloader, optimizer, criterion, device,
                n_epochs=10, scheduler=None):
    model.to(device)
    epoch_losses = []
    for epoch in range(1, n_epochs + 1):
        model.train()
        total = 0
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(X).squeeze(1)          # (batch,)
            loss   = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total += loss.item()
        if scheduler:
            scheduler.step()
        avg = total / len(dataloader)
        epoch_losses.append(avg)
        print(f'  Epoch {epoch:>2}/{n_epochs}  |  Loss: {avg:.4f}')
    return epoch_losses


# EVALUATION LOOP — do not modify.
def evaluate_neural(model, dataloader, device, name='Model'):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in dataloader:
            out  = torch.sigmoid(model(X.to(device)).squeeze(1))
            pred = (out > 0.5).long().cpu().numpy()
            all_preds.extend(pred)
            all_labels.extend(y.numpy().astype(int))
    preds, labels = np.array(all_preds), np.array(all_labels)
    print(f'\n=== {name} ===')
    print(f'  Accuracy  : {accuracy_score(labels, preds):.4f}')
    print(f'  F1        : {f1_score(labels, preds):.4f}')
    print(f'  Precision : {precision_score(labels, preds):.4f}')
    print(f'  Recall    : {recall_score(labels, preds):.4f}')
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Neg','Pos'], yticklabels=['Neg','Pos'])
    plt.title(name); plt.tight_layout(); plt.show()
    return preds, labels


# UTILITIES
def plot_losses(losses, title):
    plt.figure(figsize=(7, 3))
    plt.plot(range(1, len(losses)+1), losses, marker='o')
    plt.title(title); plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

def count_params(model):
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable parameters: {n:,}')
    return n

print('All helpers ready.')

## Stage 2A: Vanilla RNN  <font color="red">**[10 marks]**</font>

## How an RNN Processes a Review

An RNN reads the review **one token at a time**, left to right. At each step $t$:

$$h_t = \tanh(W_h \cdot h_{t-1} + W_x \cdot x_t + b)$$

- $x_t$ = embedding of the current token (a 128-dim vector)
- $h_{t-1}$ = hidden state from the previous step (memory of what came before)
- $h_t$ = updated hidden state

After reading all 400 tokens, the **final hidden state** $h_{400}$ is the model's summary of the whole review. It is then passed to a linear classifier.

**The vanishing gradient problem:** To learn from token 1, the gradient must travel back through 400 steps. Each step multiplies the gradient by the weight matrix $W_h$. If $\|W_h\| < 1$, the gradient shrinks exponentially — by the time it reaches token 1 it is essentially zero. The model cannot learn anything from distant context. You will see this in the loss curve: the RNN stalls early.

```python
self.rnn = nn.RNN(
    input_size  = embedding_dim,   # size of each input token embedding
    hidden_size = hidden_dim,      # size of hidden state h_t
    num_layers  = 2,               # stack 2 RNN layers
    batch_first = True,            # input shape: (batch, seq_len, embed_dim)
    dropout     = 0.3              # dropout between layers (only applies if num_layers > 1)
)
# rnn(embeddings) returns: (output, h_n)
#   output: (batch, seq_len, hidden_dim) — every hidden state
#   h_n:    (num_layers, batch, hidden_dim) — only the last step
# For classification we take h_n[-1]: the last layer's final hidden state.
```

### 2A.1 Build the RNN Model  <font color="red">[7 marks]</font>

Complete the `RNNSentiment` class. Architecture:

```
Input: token IDs, shape (batch, 400)
  ↓
nn.Embedding(vocab_size, 128, padding_idx=0)   ← learnable word vectors
  ↓
nn.RNN(128, 256, num_layers=2, dropout=0.3, batch_first=True)
  ↓
h_n[-1]    ← final hidden state, shape (batch, 256)
  ↓
nn.Dropout(0.5)
  ↓
nn.Linear(256, 1)   ← single logit output (no sigmoid — BCEWithLogitsLoss handles that)
```

**Step-by-step hints for `forward`:**
1. Pass `x` through the embedding layer → shape `(batch, 400, 128)`
2. Pass embeddings through the RNN → unpack as `output, h_n = self.rnn(emb)`
3. Take `h_n[-1]` → shape `(batch, 256)`
4. Apply dropout, then the linear layer

In [ ]:
class RNNSentiment(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256,
                 num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        # Define: embedding, rnn, dropout, fc
        ### CODE HERE ###

    def forward(self, x):
        # 1. Embed tokens
        ### CODE HERE ###

        # 2. Pass through RNN, get h_n
        ### CODE HERE ###

        # 3. h_n[-1] → dropout → linear
        ### CODE HERE ###

### 2A.2 Train and Evaluate  *(provided — run and observe)*

The training cell is fully written. Run it, then answer the reflection questions below.

In [ ]:
# ── PROVIDED ──────────────────────────────────────────────────────────────
rnn_model     = RNNSentiment(VOCAB_SIZE)
count_params(rnn_model)
optimizer_rnn = optim.Adam(rnn_model.parameters(), lr=1e-3)
criterion     = nn.BCEWithLogitsLoss()

print('Training RNN...')
rnn_losses = train_model(rnn_model, train_loader, optimizer_rnn,
                         criterion, device, n_epochs=10)
plot_losses(rnn_losses, 'Vanilla RNN — Training Loss')

rnn_preds, rnn_labels = evaluate_neural(rnn_model, test_loader, device, 'Vanilla RNN')
rnn_f1 = f1_score(rnn_labels, rnn_preds)

### 2A.3 Reflection  <font color="red">[3 marks]</font>

**[Your Answer]:**
1. Describe the shape of the loss curve. Does it decrease smoothly, or does it plateau? At roughly which epoch does improvement stop?
2. Compare the RNN's F1 (`rnn_f1`) to your best classical model from Part 1. Is the RNN competitive? Based on the concept box above, is this surprising?
3. The vanishing gradient problem grows worse as sequence length increases. Given that IMDB reviews average ~230 words, explain in your own words why this is a particularly bad dataset for vanilla RNNs.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

3. ***your answer here***

## Stage 2B: LSTM  <font color="red">**[10 marks]**</font>

## How the LSTM Fixes the Vanishing Gradient

The LSTM adds a second vector: the **cell state** $c_t$. Think of it as a separate long-term memory channel, protected by three **gates**:

| Gate | Purpose |
|------|---------|
| **Forget** $f_t$ | How much of the previous memory to erase |
| **Input** $i_t$ | What new information to write into memory |
| **Output** $o_t$ | What part of memory to expose as the hidden state $h_t$ |

$$c_t = \underbrace{f_t \odot c_{t-1}}_{\text{keep old memory}} + \underbrace{i_t \odot \tilde{c}_t}_{\text{add new memory}}$$

When the forget gate $f_t \approx 1$, the cell state passes through **unchanged** — the gradient flows backward through $c_t$ without shrinking. This is the gradient highway that vanilla RNNs lack.

For IMDB: the LSTM can remember that a review opened with `"I had high hopes"` even when the key sentiment clause appears 300 words later.

```python
self.lstm = nn.LSTM(
    input_size=embedding_dim, hidden_size=hidden_dim,
    num_layers=2, batch_first=True, dropout=0.3
)
# LSTM returns: (output, (h_n, c_n))   ← note the nested tuple!
# Unpack like this:
output, (h_n, c_n) = self.lstm(emb)
# h_n[-1] is still the last layer's final hidden state — same as the RNN.
```

### 2B.1 Build the LSTM Model  <font color="red">[7 marks]</font>

The architecture is **identical to the RNN** — only replace `nn.RNN` with `nn.LSTM`:

```
Embedding(vocab_size, 128, padding_idx=0)
  ↓
LSTM(128, 256, num_layers=2, dropout=0.3, batch_first=True)
  ↓
h_n[-1]    ← (batch, 256)
  ↓
Dropout(0.5)
  ↓
Linear(256, 1)
```

**The only code difference from the RNN:**
- Use `nn.LSTM` instead of `nn.RNN`
- Unpack as `_, (h_n, _) = self.lstm(emb)` — the second return value is a tuple `(h_n, c_n)`

In [ ]:
class LSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256,
                 num_layers=2, dropout=0.3, pad_idx=0):
        super().__init__()
        # Define: embedding, lstm, dropout, fc
        ### CODE HERE ###

    def forward(self, x):
        # 1. Embed tokens
        ### CODE HERE ###

        # 2. LSTM — unpack (output, (h_n, c_n))
        ### CODE HERE ###

        # 3. h_n[-1] → dropout → linear
        ### CODE HERE ###

In [ ]:
# ── PROVIDED ──────────────────────────────────────────────────────────────
lstm_model     = LSTMSentiment(VOCAB_SIZE)
count_params(lstm_model)
optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=1e-3)

print('Training LSTM...')
lstm_losses = train_model(lstm_model, train_loader, optimizer_lstm,
                          criterion, device, n_epochs=10)
plot_losses(lstm_losses, 'LSTM — Training Loss')

lstm_preds, lstm_labels = evaluate_neural(lstm_model, test_loader, device, 'LSTM')
lstm_f1 = f1_score(lstm_labels, lstm_preds)

### 2B.2 Reflection  <font color="red">[3 marks]</font>

**[Your Answer]:**
1. Compare the RNN and LSTM loss curves. Which converges further and more smoothly? Connect your observation to the gate mechanism described above.
2. How much did F1 improve (`rnn_f1` → `lstm_f1`)? Is this a small or large jump for such a minimal code change?
3. The LSTM still reads tokens **left to right only**. Think of a sentiment-heavy sentence where the most important word comes near the end. Why might this still be a problem?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

3. ***your answer here***

## Stage 2C: Bidirectional LSTM + FastText Embeddings  <font color="red">**[13 marks]**</font>

## Two Improvements at Once

### Bidirectional LSTM

A standard LSTM reads left-to-right. Consider:

> *"A film that, despite its slow start, delivers a genuinely moving finale."*

When the LSTM reaches `"slow"` it has not yet seen `"moving"` or `"finale"`. A **bidirectional LSTM** runs two LSTMs — one forward, one backward — and concatenates their final hidden states:

```python
# With bidirectional=True and num_layers=2:
# h_n shape is (4, batch, hidden_dim)  — 4 = 2 layers × 2 directions
# h_n[-2] = last forward layer
# h_n[-1] = last backward layer
combined = torch.cat([h_n[-2], h_n[-1]], dim=1)  # (batch, hidden_dim * 2)
```
The hidden dimension **doubles**: 256 → 512.

### FastText Pre-trained Embeddings

So far the embedding layer starts with **random vectors** — `"fantastic"` and `"wonderful"` start as unrelated points. The model must learn their similarity purely from sentiment labels.

**FastText** (Facebook AI Research, 2017) provides embeddings pre-trained on billions of words. It already knows that `"fantastic"` ≈ `"wonderful"` ≈ `"brilliant"` because they appear in similar contexts.

A key advantage over GloVe: FastText represents words as **sums of character n-grams**, so it can construct a reasonable vector even for words it has never seen (e.g. typos, rare words).

Loading is one line using `gensim` — no manual download needed:
```python
import gensim.downloader as api
ft_model = api.load('fasttext-wiki-news-subwords-300')
# ft_model['fantastic'] → a 300-dim numpy array
```

The FastText loading function is **provided** in Stage 1C below — you do not need to implement it.

### Stage 1C: Load FastText Embeddings  *(provided — run once)*

This cell downloads and caches FastText embeddings using `gensim`. It runs once and caches locally — subsequent runs are fast.

> **Note:** The download is ~1 GB. On Google Colab this takes ~2 minutes. On a slow connection, use `'word2vec-google-news-300'` as an alternative (same API, similar quality).

In [ ]:
# ── PROVIDED ──────────────────────────────────────────────────────────────
import gensim.downloader as api

print('Loading FastText embeddings (downloads once, ~1 GB, then cached)...')
ft_model = api.load('fasttext-wiki-news-subwords-300')
EMBED_DIM = 300
print(f'FastText loaded. Vocabulary: {len(ft_model):,} words, dim: {EMBED_DIM}')

# Build embedding matrix aligned with our word2idx vocabulary
def build_embedding_matrix(word2idx, ft_model, dim):
    matrix = np.random.uniform(-0.1, 0.1, (len(word2idx), dim)).astype(np.float32)
    matrix[0] = 0.0   # PAD token = zero vector
    found = 0
    for word, idx in word2idx.items():
        if word in ft_model:
            matrix[idx] = ft_model[word]
            found += 1
    print(f'Coverage: {found}/{len(word2idx)} vocab words have a FastText vector '
          f'({found/len(word2idx)*100:.1f}%)')
    return torch.FloatTensor(matrix)

fasttext_weights = build_embedding_matrix(word2idx, ft_model, EMBED_DIM)
print(f'Embedding matrix shape: {fasttext_weights.shape}')

### 2C.1 Build the BiLSTM Model  <font color="red">[8 marks]</font>

```
Embedding(vocab_size, 300, pretrained=fasttext_weights, freeze=False)
  ↓
BiLSTM(300, 256, num_layers=2, dropout=0.3, bidirectional=True)
  ↓
concat [h_n[-2] ; h_n[-1]]   ← shape (batch, 512)
  ↓
Dropout(0.5)
  ↓
Linear(512, 256) → ReLU
  ↓
Linear(256, 1)
```

**Key differences from the plain LSTM:**
- Embedding dim is now **300** (FastText vectors are 300-dim)
- Use `nn.Embedding.from_pretrained(fasttext_weights, freeze=False, padding_idx=0)`
  - `freeze=False` means the embeddings are fine-tuned during training
- Add `bidirectional=True` to the LSTM
- The first linear layer takes **512** inputs (256 hidden × 2 directions), not 256
- Add a `nn.ReLU()` between the two linear layers

In [ ]:
class BiLSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, hidden_dim=256,
                 num_layers=2, dropout=0.3, pad_idx=0,
                 pretrained_embeddings=None):
        super().__init__()

        # Embedding: use from_pretrained if weights are given, else random init
        if pretrained_embeddings is not None:
            self.embedding = nn.Embedding.from_pretrained(
                pretrained_embeddings, freeze=False, padding_idx=pad_idx
            )
        else:
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        # Bidirectional LSTM
        ### CODE HERE ###

        # Classifier head — input is 512 (hidden*2) because bidirectional
        ### CODE HERE ###

    def forward(self, x):
        # 1. Embed tokens
        ### CODE HERE ###

        # 2. BiLSTM — unpack (output, (h_n, c_n))
        ### CODE HERE ###

        # 3. Concatenate h_n[-2] (forward) and h_n[-1] (backward)
        ### CODE HERE ###

        # 4. dropout → fc1 → ReLU → fc2
        ### CODE HERE ###

In [ ]:
# ── PROVIDED ──────────────────────────────────────────────────────────────
bilstm_model   = BiLSTMSentiment(VOCAB_SIZE, pretrained_embeddings=fasttext_weights)
count_params(bilstm_model)
optimizer_bi   = optim.Adam(bilstm_model.parameters(), lr=5e-4)

print('Training BiLSTM + FastText...')
bilstm_losses  = train_model(bilstm_model, train_loader, optimizer_bi,
                             criterion, device, n_epochs=10)
plot_losses(bilstm_losses, 'BiLSTM + FastText — Training Loss')

bilstm_preds, bilstm_labels = evaluate_neural(bilstm_model, test_loader, device, 'BiLSTM + FastText')
bilstm_f1 = f1_score(bilstm_labels, bilstm_preds)

### 2C.2 Reflection  <font color="red">[5 marks]</font>

**[Your Answer]:**
1. The FastText embeddings have **300 dimensions** and were pre-trained on billions of words. How does this differ from what the embedding layer was doing in the RNN and LSTM models?
2. What does `freeze=False` mean? What would happen if you set `freeze=True` instead — what is the trade-off?
3. Did the BiLSTM + FastText improve over the plain LSTM? Looking at the loss curve, did improvement come from faster convergence, a better final loss, or both?
4. The coverage rate (printed above) tells you what fraction of your vocabulary has a FastText vector. What happens to words that **don't** have a FastText vector? Is this a problem?
5. FastText uses **character n-grams** to build vectors for unknown words. Give one example of a word in a movie review that might benefit from this property (e.g. a typo, an unusual word form).

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

3. ***your answer here***

4. ***your answer here***

5. ***your answer here***

## Stage 2D: Cross-Attention Classifier  <font color="red">**[12 marks]**</font>

## Attention: Reading the Whole Review at Once

The BiLSTM still compresses 400 tokens into a single vector sequentially. No matter how good the gates are, long-range dependencies remain hard.

**Attention** removes the sequential bottleneck entirely. Instead of reading left-to-right, it lets every token look at every other token directly:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V$$

- $Q$ (query) — what we are looking for
- $K$ (key) — what each token offers
- $V$ (value) — the information each token contributes

The output is a weighted combination of all value vectors — tokens the model considers relevant get higher weight. Token 1 and Token 400 are **equidistant**: no forgetting over distance.

### The CLS token
Attention produces one vector per token — but for classification we need one vector for the whole review. The solution: prepend a learnable **`[CLS]`** token. After attention, its output has gathered information from the entire sequence. We classify from that single vector.

### Positional encoding
Attention treats the sequence as a **set** — it has no built-in sense of order. `"I loved it"` and `"it loved I"` would produce identical attention scores. **Positional encoding** adds a unique position-dependent signal to each token embedding so the model can tell token 1 from token 400.

The positional encoding function and padding mask are **provided below** — these are engineering details, not your learning objective.

### Stage 1D: Attention Utilities  *(provided — run once)*

In [ ]:
# ── PROVIDED ──────────────────────────────────────────────────────────────
# POSITIONAL ENCODING
# Adds a unique sinusoidal pattern to each position so the model knows token order.
# Returns shape (1, max_len, d_model) — added to the token embedding batch.
def positional_encoding(max_len, d_model):
    pe  = torch.zeros(max_len, d_model)
    pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div[:d_model // 2])
    return pe.unsqueeze(0)


# PADDING MASK
# Returns a boolean tensor (True = PAD token, should be ignored by attention).
# The +1 accounts for the CLS token prepended before the sequence.
def make_pad_mask(x):
    B   = x.shape[0]
    pad = (x == 0)                                              # True where token = PAD
    cls = torch.zeros(B, 1, dtype=torch.bool, device=x.device) # CLS is never PAD
    return torch.cat([cls, pad], dim=1)                         # (batch, seq+1)


# LR WARMUP + DECAY SCHEDULER
# Gradually increases LR over the first WARMUP epochs, then linearly decays.
# Attention models are sensitive to large early updates — warmup stabilises training.
def make_scheduler(optimizer, n_epochs, warmup_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        return max(0.0, (n_epochs - epoch) / (n_epochs - warmup_epochs))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print('Attention utilities ready.')

### 2D.1 Build the Cross-Attention Model  <font color="red">[7 marks]</font>

```
Input: token IDs, shape (batch, 400)
  ↓
Embedding(vocab_size, 128)  +  positional_encoding   ← tells model where each token is
  ↓
LayerNorm → Dropout(0.1)
  ↓
Prepend [CLS] token   → sequence shape becomes (batch, 401, 128)
  ↓
nn.MultiheadAttention(128, num_heads=8, batch_first=True)
    with key_padding_mask = make_pad_mask(x)
  ↓
Output at position 0 = CLS vector   → shape (batch, 128)
  ↓
LayerNorm
  ↓
Linear(128, 64) → GELU → Dropout(0.3)
  ↓
Linear(64, 1)
```

**Step-by-step hints for `__init__`:**
- `self.embedding` — `nn.Embedding(vocab_size, 128, padding_idx=0)`
- `self.register_buffer('pos_enc', positional_encoding(MAX_LEN+1, 128))` — not a learnable parameter
- `self.cls_token = nn.Parameter(torch.randn(1, 1, 128))` — one learnable vector per model
- `self.norm1` — `nn.LayerNorm(128)` before attention
- `self.drop1` — `nn.Dropout(0.1)`
- `self.attention` — `nn.MultiheadAttention(128, num_heads=8, dropout=0.1, batch_first=True)`
- `self.norm2` — `nn.LayerNorm(128)` after attention
- `self.classifier` — `nn.Sequential(Linear(128,64), nn.GELU(), nn.Dropout(0.3), Linear(64,1))`
- `self.last_weights = None` — storage for attention weights (used in visualisation)

**Step-by-step hints for `forward`:**
1. Embed + add positional encoding: `emb = self.embedding(x) + self.pos_enc[:, 1:x.shape[1]+1, :]`
2. Apply `norm1` and `drop1`
3. Expand CLS: `cls = self.cls_token.expand(x.shape[0], -1, -1)`
4. Prepend: `seq = torch.cat([cls, emb], dim=1)` → shape `(batch, 401, 128)`
5. Attention: `out, w = self.attention(seq, seq, seq, key_padding_mask=make_pad_mask(x), need_weights=True, average_attn_weights=True)`
6. Save weights: `self.last_weights = w.detach().cpu()`
7. CLS output: `cls_repr = self.norm2(out[:, 0, :])` → shape `(batch, 128)`
8. Classify: `return self.classifier(cls_repr)`

In [ ]:
class CrossAttnSentiment(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, num_heads=8,
                 dropout=0.1, pad_idx=0):
        super().__init__()
        ### CODE HERE ###

    def forward(self, x):
        ### CODE HERE ###

In [ ]:
# ── PROVIDED ──────────────────────────────────────────────────────────────
N_EPOCHS = 10

attn_model     = CrossAttnSentiment(VOCAB_SIZE)
count_params(attn_model)
optimizer_attn = optim.Adam(attn_model.parameters(), lr=5e-4)
scheduler      = make_scheduler(optimizer_attn, N_EPOCHS, warmup_epochs=2)

print('Training Cross-Attention model...')
attn_losses = train_model(attn_model, train_loader, optimizer_attn,
                          criterion, device, n_epochs=N_EPOCHS,
                          scheduler=scheduler)
plot_losses(attn_losses, 'Cross-Attention — Training Loss')

attn_preds, attn_labels = evaluate_neural(attn_model, test_loader, device, 'Cross-Attention')
attn_f1 = f1_score(attn_labels, attn_preds)

### 2D.2 Attention Visualisation  *(provided — run and interpret)*

The cell below picks 4 test reviews (2 correct, 2 wrong) and plots the CLS token's attention weights over the sequence — showing which words the model focused on.

**[Your Answer]:**  <font color="red">[2 marks]</font>

- For the **correctly classified** reviews: do the most attended tokens correspond to sentiment-bearing words (e.g. `"brilliant"`, `"terrible"`)?
- For the **misclassified** reviews: what did the model attend to instead? Can you see why it was confused?

*Write your answers here:*

***your answer here***

In [ ]:
# ── PROVIDED ──────────────────────────────────────────────────────────────
attn_model.eval()

groups = [
    ('Correct — Positive', np.where((attn_labels==1) & (attn_preds==1))[0]),
    ('Correct — Negative', np.where((attn_labels==0) & (attn_preds==0))[0]),
    ('Wrong — False Negative', np.where((attn_labels==1) & (attn_preds==0))[0]),
    ('Wrong — False Positive', np.where((attn_labels==0) & (attn_preds==1))[0]),
]

fig, axes = plt.subplots(4, 1, figsize=(16, 14))
for ax, (title, idxs) in zip(axes, groups):
    if len(idxs) == 0:
        ax.set_title(f'{title} — no examples found'); continue
    X_s, _ = test_ds[idxs[0]]
    with torch.no_grad():
        attn_model(X_s.unsqueeze(0).to(device))
    weights = attn_model.last_weights[0, 0, 1:].numpy()  # CLS row, skip CLS col
    tokens  = [idx2word.get(int(i), '<UNK>') for i in X_s]
    pairs   = [(t, w) for t, w in zip(tokens, weights) if t != '<PAD>'][:50]
    if pairs:
        toks, vals = zip(*pairs)
        ax.bar(range(len(toks)), vals, color='steelblue')
        ax.set_xticks(range(len(toks)))
        ax.set_xticklabels(toks, rotation=70, ha='right', fontsize=7)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('Attention weight')
plt.suptitle('CLS Attention Weights — First 50 Tokens', fontsize=12)
plt.tight_layout(); plt.show()

### 2D.3 Reflection  <font color="red">[3 marks]</font>

**[Your Answer]:**
1. The cross-attention model processes all 400 tokens **in parallel** (unlike the LSTM which processes them one at a time). What is one practical advantage of this for training on a GPU?
2. Why is **positional encoding** necessary for the attention model? What would happen if you removed it?
3. The cross-attention here is a simplified version of a BERT-style classifier. What is the main thing a full BERT model has that this model does not?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

3. ***your answer here***

## Stage 3: Full Comparison  <font color="red">**[10 marks]**</font>

### 3.1 Master Comparison Table  <font color="red">[3 marks]</font>

Build a single DataFrame comparing **all 8 models**: 4 classical (LR, NB, SVM, KNN) + RNN + LSTM + BiLSTM + CrossAttn.

Columns: Model | Accuracy | Precision | Recall | F1 | Parameters

For classical models, set Parameters = `"N/A"`. Sort by F1 descending.

In [ ]:
### CODE HERE ###

### 3.2 All Neural Loss Curves  <font color="red">[3 marks]</font>

Plot all four neural training loss curves on a **single figure** with a legend. Use different line styles for each model.

In [ ]:
### CODE HERE ###

### 3.3 Error Analysis  <font color="red">[4 marks]</font>

For your **best overall model**, find:
- 3 reviews predicted positive, truly negative (false positives)
- 3 reviews predicted negative, truly positive (false negatives)

Print the first 60 words of each. For each, write one sentence explaining what misled the model.

In [ ]:
### CODE HERE ###

## Final Reflection  <font color="red">**[10 marks]**</font>

**[Your Answer]:** Answer each question. Cite specific F1 numbers.

1. **One-hot ceiling:** At what F1 did your classical models plateau? Why does the absence of frequency and rarity information hurt so much? Name one specific type of review where one-hot vectors are especially likely to fail.

2. **RNN → LSTM gap:** How large was the F1 jump? Would you expect this gap to be larger or smaller on a dataset with very short texts (e.g. tweets, ~10 words)? Why?

3. **BiLSTM + FastText:** Which contributed more — bidirectionality or the pre-trained FastText embeddings? How can you tell from your results?

4. **Attention:** Did the cross-attention model beat the BiLSTM? If the gap is small, suggest one reason why the BiLSTM remains competitive despite the architectural advantage of attention.

5. **Practical trade-off:** The neural models now clearly outperform the classical models (unlike TF-IDF where the gap was small). Does this change your deployment decision? What additional factors beyond accuracy would you consider?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

3. ***your answer here***

4. ***your answer here***

5. ***your answer here***

---
## Mark Summary

| Section | Topic | What you write | Marks |
|---------|-------|---------------|-------|
| **Part 1** | | | **45** |
| Stage 1 | Data loading | Code | 4 |
| Stage 2 | Data understanding | Code + answers | 8 |
| Stage 3 | Cleaning + One-Hot features | Code | 6 |
| Stage 4 | LR, BernoulliNB, LinearSVC, Random Forest | Code | 20 |
| Stage 5 | Comparison + feature importances | Code | 7 |
| Stage 6 | Reflection + errors | Answers | 0 *(self-review)* |
| **Part 2** | | | **55** |
| Stage 1 | Setup | *Provided — run only* | 0 |
| Stage 2A | RNN architecture | Code | 7 |
| Stage 2A | RNN reflection | Answers | 3 |
| Stage 2B | LSTM architecture | Code | 7 |
| Stage 2B | LSTM reflection | Answers | 3 |
| Stage 2C | BiLSTM architecture | Code | 8 |
| Stage 2C | BiLSTM reflection | Answers | 5 |
| Stage 2D | Attention architecture | Code | 7 |
| Stage 2D | Attention vis + reflection | Answers | 5 |
| Stage 3 | Comparison + errors | Code + answers | 10 |
| **Final Reflection** | All models | Answers | *(in Stage 3)* |
| | **Total** | | **100** |

---
*End of Assignment*